Elaboró:

ROJAS MARTINEZ JONATHAN FRANCISCO

# Algoritmos genéticos

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random

In [2]:
# Valor en bots - Valor en decimal - Aptitud - probabilidad de seleccion 

In [3]:
# Función para generar un numero binario de longitud de n_bits
def num_bin_ale(n_bits):
    return [random.randint(0, 1) for _ in range(n_bits)]

In [4]:
# Función para convertir un numero binario a decimal
def decodificar_binario(bits, a, b):
    n = len(bits)
    # Convertir lista de bits a string y luego a entero decimal
    decimal = int("".join(str(bit) for bit in bits), 2)
    # Interpolar en el intervalo [a, b] (normalizar)
    return a + decimal * (b - a) / ((2**n) - 1)

In [5]:
# Funcion de aptitud
def fitness_function(x):
    return -x**3 + 60*x**2 + 15000

In [6]:
# Seleccion de los mas fuertes
def seleccion(valores, num_seleccionados):
    # ordenamos de mayor a menor la parte de aptitudes y obtenemos los indices
    indices_ordenados = valores[:, 2].argsort()[::-1]
    valores = valores[indices_ordenados]
    # seleccionamos los mejores individuos
    seleccionados = [valores[i] for i in range(num_seleccionados)]
    return seleccionados

In [7]:
# Cruza entre 2 individuos
def cruza(in1, in2, n_bits):
    # 1. Hacemos copias independientes para no alterar a los padres originales
    hijo1 = in1.copy()
    hijo2 = in2.copy()
    
    # Escogemos una cantidad de puntos de cruza aleatorios
    puntos_cruza = random.randint(1, n_bits - 1) # Sin el 0 porque queremos 1 o mas puntos de cruza
    
    # Las posiciones empezando de atrás para adelante
    for punto in range(1, puntos_cruza + 1):
        # Intercambiamos los bits aleatoriamente entre los dos que tengan
        accion = random.choice([0, 1])
        if accion == 0:
            # Modificamos solo a los hijos
            hijo1[-punto], hijo2[-punto] = hijo2[-punto], hijo1[-punto]
            
    # Retornamos las nuevas listas
    return hijo1, hijo2

In [8]:
# Ruleta de nuevo nacimiento 
# Tenemos n individuos cada uno con una probablidad de ser seleccionado 
def ruleta_funcrandom(valores, num_seleccionados):
    # Seleccionamos un individuo basado en las probabilidades de seleccion
    seleccionados = np.random.choice(valores[:, 0], p=valores[:, 3].astype(float), size=num_seleccionados)
    return seleccionados

In [9]:
def mutacion(individuo):
    posicion = random.randrange(len(individuo))
    individuo[posicion] = 1 - individuo[posicion]  # Cambia el bit (0 a 1 o 1 a 0)
    return individuo

In [10]:
def creacion_de_generacion(poblacion, n_bits):
    return [num_bin_ale(n_bits) for _ in range(poblacion)]

In [11]:
# Selección de los mejores:
def probabilidades_de_seleccion(aptitudes):
    total_aptitud = sum(aptitudes)
    probabilidades = aptitudes / total_aptitud
    return probabilidades

In [12]:
def crear_datos_gen(poblacion, intervalo):
    temp = []
    for individuo in poblacion:
        valor_decimal = decodificar_binario(individuo, intervalo[0], intervalo[1])
        aptitud = fitness_function(valor_decimal)
        temp.append([individuo, valor_decimal, aptitud])
    temp = np.array(temp, dtype=object)
    temp = np.append(temp, probabilidades_de_seleccion(temp[:, 2]).reshape(-1, 1), axis=1)
    # poblacion = np.array(temp, dtype=object)
    return temp

In [13]:
# Valores para el problema
n_bits = 6
intervalo = [0, 63]
poblacion = 6

In [14]:
primer_generacion = creacion_de_generacion(poblacion, n_bits)

In [15]:
# Creacion de los datos de la primer generacion
primer_generacion = crear_datos_gen(primer_generacion, intervalo)

In [16]:
primer_generacion

array([[list([1, 1, 1, 1, 0, 0]), 60.0, 15000.0, 0.09660653446599128],
       [list([1, 1, 1, 0, 1, 1]), 59.0, 18481.0, 0.11902569089773232],
       [list([0, 1, 0, 0, 0, 0]), 16.0, 26264.0, 0.16915160141431967],
       [list([0, 1, 1, 0, 1, 0]), 26.0, 37984.0, 0.2446335070104142],
       [list([0, 0, 1, 0, 0, 1]), 9.0, 19131.0, 0.12321197405792528],
       [list([1, 1, 0, 0, 1, 1]), 51.0, 38409.0, 0.24737069215361726]],
      dtype=object)

In [17]:
# Nacimiento por ruleta
segunda_generacion = ruleta_funcrandom(primer_generacion, poblacion)

In [18]:
segunda_generacion

array([list([0, 1, 1, 0, 1, 0]), list([1, 1, 0, 0, 1, 1]),
       list([0, 1, 1, 0, 1, 0]), list([1, 1, 1, 1, 0, 0]),
       list([1, 1, 1, 0, 1, 1]), list([1, 1, 0, 0, 1, 1])], dtype=object)

In [19]:
segunda_generacion = crear_datos_gen(segunda_generacion, intervalo)
print("Segunda generación:")
print(segunda_generacion)

Segunda generación:
[[list([0, 1, 1, 0, 1, 0]) 26.0 37984.0 0.20392232655274417]
 [list([1, 1, 0, 0, 1, 1]) 51.0 38409.0 0.20620399748747764]
 [list([0, 1, 1, 0, 1, 0]) 26.0 37984.0 0.20392232655274417]
 [list([1, 1, 1, 1, 0, 0]) 60.0 15000.0 0.08052956240235791]
 [list([1, 1, 1, 0, 1, 1]) 59.0 18481.0 0.09921778951719842]
 [list([1, 1, 0, 0, 1, 1]) 51.0 38409.0 0.20620399748747764]]


In [20]:
num_seleccionados = 3
seleccionados = seleccion(segunda_generacion, num_seleccionados)
print("Seleccionados para cruza:")
print(seleccionados)

Seleccionados para cruza:
[array([list([1, 1, 0, 0, 1, 1]), 51.0, 38409.0, 0.20620399748747764],
      dtype=object), array([list([1, 1, 0, 0, 1, 1]), 51.0, 38409.0, 0.20620399748747764],
      dtype=object), array([list([0, 1, 1, 0, 1, 0]), 26.0, 37984.0, 0.20392232655274417],
      dtype=object)]


In [21]:
# Cruzamos entre los mejores:
tercera_generacion = []
for i in range(0, len(seleccionados)):
    for j in range(i + 1, len(seleccionados)):
        in1, in2 = seleccionados[i][0], seleccionados[j][0]
        hijo1, hijo2 = cruza(in1.copy(), in2.copy(), n_bits)
        print(f"Cruza entre {i+1} -- {in1} y {j+1} -- {in2} da como resultado: {hijo1} y {hijo2}")
        tercera_generacion.append(hijo1)
        tercera_generacion.append(hijo2)

Cruza entre 1 -- [1, 1, 0, 0, 1, 1] y 2 -- [1, 1, 0, 0, 1, 1] da como resultado: [1, 1, 0, 0, 1, 1] y [1, 1, 0, 0, 1, 1]
Cruza entre 1 -- [1, 1, 0, 0, 1, 1] y 3 -- [0, 1, 1, 0, 1, 0] da como resultado: [1, 1, 1, 0, 1, 0] y [0, 1, 0, 0, 1, 1]
Cruza entre 2 -- [1, 1, 0, 0, 1, 1] y 3 -- [0, 1, 1, 0, 1, 0] da como resultado: [1, 1, 0, 0, 1, 1] y [0, 1, 1, 0, 1, 0]


In [22]:
# Mutamos uno de la tercera generación:
individuo_a_mutar = random.choice(tercera_generacion)
print(f"Individuo antes de mutar: {individuo_a_mutar}")
individuo_mutado = mutacion(individuo_a_mutar.copy())
print(f"Individuo después de mutar: {individuo_mutado}")

Individuo antes de mutar: [0, 1, 1, 0, 1, 0]
Individuo después de mutar: [0, 1, 1, 0, 0, 0]


In [23]:
print("Tercera generación (hijos de la cruza):")
print(tercera_generacion)   

Tercera generación (hijos de la cruza):
[[1, 1, 0, 0, 1, 1], [1, 1, 0, 0, 1, 1], [1, 1, 1, 0, 1, 0], [0, 1, 0, 0, 1, 1], [1, 1, 0, 0, 1, 1], [0, 1, 1, 0, 1, 0]]


In [25]:
# --- 1. PREPARACIÓN Y PARÁMETROS INICIALES ---
n_bits = 6
intervalo = [0, 63]
tamano_poblacion = 6
generaciones_totales = 3  # Puedes aumentar esto a 50 o 100 después
probabilidad_mutacion = 0.15 # 15% de probabilidad de que un hijo mute

print("==================================================")
print("       INICIANDO ALGORITMO GENÉTICO")
print("==================================================\n")

# Inicializamos la primera generación (solo cadenas de bits)
poblacion_actual = creacion_de_generacion(tamano_poblacion, n_bits)

# --- 2. EL BUCLE DE GENERACIONES ---
for gen in range(generaciones_totales):
    print(f"--- GENERACIÓN {gen + 1} ---")
    
    # A) EVALUACIÓN
    # Convertimos los bits en datos útiles (decimal, aptitud, probabilidad)
    datos_poblacion = crear_datos_gen(poblacion_actual, intervalo)
    
    print("1. Evaluación:")
    for ind in datos_poblacion:
        # Imprimimos de forma limpia: Bits | Decimal | Aptitud | Probabilidad
        print(f"   Bits: {ind[0]} | Dec: {ind[1]:.2f} | Aptitud: {ind[2]:.2f} | Prob: {ind[3]:.4f}")

    # B) SELECCIÓN
    # Usamos tu función de ruleta para escoger quiénes se reproducen
    # La ruleta ya tiende a escoger a los de mayor probabilidad
    padres_seleccionados = ruleta_funcrandom(datos_poblacion, tamano_poblacion)
    print("\n2. Selección (Padres elegidos para reproducirse):")
    for p in padres_seleccionados:
        print(f"   {p}")

    # C) CRUZA
    siguiente_generacion = []
    print("\n3. Cruza:")
    # Tomamos a los padres de 2 en 2
    for i in range(0, tamano_poblacion, 2):
        padre1 = padres_seleccionados[i].copy()
        # Evitamos error de índice si la población es impar
        padre2 = padres_seleccionados[i+1].copy() if (i+1) < tamano_poblacion else padres_seleccionados[0].copy()
        
        hijo1, hijo2 = cruza(padre1, padre2, n_bits)
        siguiente_generacion.extend([hijo1, hijo2])
        print(f"   Cruzando {padre1} y {padre2} -> Hijos: {hijo1}, {hijo2}")

    # D) MUTACIÓN
    print("\n4. Mutación:")
    for i in range(len(siguiente_generacion)):
        # Solo mutamos si caemos dentro de la probabilidad
        if random.random() < probabilidad_mutacion:
            antes_mutar = siguiente_generacion[i].copy()
            siguiente_generacion[i] = mutacion(siguiente_generacion[i])
            print(f"   ¡Mutación! El individuo {i} cambió de {antes_mutar} a {siguiente_generacion[i]}")
    
    # E) REEMPLAZO
    # Los hijos se convierten en la población actual para el siguiente ciclo
    poblacion_actual = siguiente_generacion
    print(f"--------------------------------------------------\n")

# --- 3. RESULTADO FINAL ---
print("================== RESULTADO FINAL ==================")
datos_finales = crear_datos_gen(poblacion_actual, intervalo)
# Buscamos al individuo con la aptitud más alta (índice 2 de tu arreglo)
mejor_individuo = max(datos_finales, key=lambda x: x[2])

print(f"El mejor individuo encontrado tras {generaciones_totales} generaciones es:")
print(f"Bits: {mejor_individuo[0]}")
print(f"Valor Decimal: {mejor_individuo[1]:.2f}")
print(f"Aptitud Alcanzada: {mejor_individuo[2]:.2f}")

       INICIANDO ALGORITMO GENÉTICO

--- GENERACIÓN 1 ---
1. Evaluación:
   Bits: [0, 0, 1, 1, 1, 0] | Dec: 14.00 | Aptitud: 24016.00 | Prob: 0.1690
   Bits: [1, 1, 0, 1, 1, 0] | Dec: 54.00 | Aptitud: 32496.00 | Prob: 0.2286
   Bits: [0, 1, 0, 1, 1, 0] | Dec: 22.00 | Aptitud: 33392.00 | Prob: 0.2349
   Bits: [0, 0, 0, 0, 1, 1] | Dec: 3.00 | Aptitud: 15513.00 | Prob: 0.1091
   Bits: [0, 0, 1, 0, 0, 1] | Dec: 9.00 | Aptitud: 19131.00 | Prob: 0.1346
   Bits: [0, 0, 0, 1, 1, 1] | Dec: 7.00 | Aptitud: 17597.00 | Prob: 0.1238

2. Selección (Padres elegidos para reproducirse):
   [1, 1, 0, 1, 1, 0]
   [0, 0, 0, 1, 1, 1]
   [0, 1, 0, 1, 1, 0]
   [0, 0, 0, 0, 1, 1]
   [0, 0, 1, 1, 1, 0]
   [0, 0, 0, 1, 1, 1]

3. Cruza:
   Cruzando [1, 1, 0, 1, 1, 0] y [0, 0, 0, 1, 1, 1] -> Hijos: [1, 1, 0, 1, 1, 1], [0, 0, 0, 1, 1, 0]
   Cruzando [0, 1, 0, 1, 1, 0] y [0, 0, 0, 0, 1, 1] -> Hijos: [0, 1, 0, 1, 1, 0], [0, 0, 0, 0, 1, 1]
   Cruzando [0, 0, 1, 1, 1, 0] y [0, 0, 0, 1, 1, 1] -> Hijos: [0, 0, 1, 1, 1, 

- En general sacar las graficas de todos los algoritmos 
- Cauchy - graficas
- investigar para un polinomio como el de la clase resolverlo analíticamente para el intervalo que propongamos.
- tablas para todos los algoritmos
- el del lineal sacar las gráficas para ver las zonas y lineas de intersección, areas y linea punteada la de aptitud
- recodico mejorar la legibilidad del código y poner las funciones de temperatura junto con las graficas de donde van los puntos.
- Documentación de todos los códigos y como funcionan (2 - 3 hojas)